## 🚀 Advanced MLflow: Deep Learning with Hyperparameter Optimization

### What You'll Learn:
This is the **most advanced notebook** in the project! We'll use:
- **Keras/TensorFlow**: Build a deep learning neural network
- **Hyperopt**: Intelligently search for best hyperparameters (smarter than grid search!)
- **MLflow**: Track every experiment and compare results
- **Wine Quality Prediction**: Predict wine quality from chemical properties

### Key Features:
1. **Neural Networks instead of classical ML**: Use deep learning for complex patterns
2. **Hyperopt (not GridSearchCV)**: Uses Bayesian optimization - tests fewer combinations but smarter!
3. **Nested Runs**: Each hyperopt iteration creates a sub-run tracked in MLflow
4. **Model Deployment**: Load and use the best model for predictions

### Real-World Significance:
This workflow is exactly what companies use to:
- Train neural networks efficiently
- Track thousands of experiments automatically
- Deploy models to production
- A/B test different model versions

In [ ]:
# ============================================================================
# STEP 1: Import All Required Libraries
# ============================================================================

# Deep Learning
import keras                                    # Neural network library
import tensorflow as tf                        # TensorFlow backend
print(f"✅ Keras version: {keras.__version__}")

# Scientific Computing
import numpy as np                              # Numerical operations
import pandas as pd                             # Data manipulation

# Hyperparameter Optimization
from hyperopt import STATUS_OK, Trials, fmin, hp, tpe  
# - STATUS_OK: Status code for successful run
# - Trials: Stores all hyperopt trials/runs
# - fmin: Function minimization (find best hyperparams)
# - hp: Hyperparameter space definition
# - tpe: Tree-structured Parzen Estimator (smart search algorithm)

# Model Evaluation
from sklearn.metrics import mean_squared_error  # Calculate model error
from sklearn.model_selection import train_test_split  # Split data

# MLflow Tracking
import mlflow                                  # Experiment tracking
from mlflow.models import infer_signature     # Create model signature

print("✅ All libraries imported successfully!")

d:\anaconda\envs\mlflow_env\lib\site-packages\hyperopt\atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [ ]:
# ============================================================================
# STEP 2: Load Wine Quality Dataset from MLflow Repository
# ============================================================================
# This dataset contains physicochemical properties of Portuguese white wines
# Features: Fixed acidity, volatile acidity, citric acid, alcohol, sulfates, etc.
# Target: Quality score (3-9, higher = better quality)

print("📥 Loading Wine Quality dataset from MLflow repository...\n")

# Load from URL (MLflow's official test dataset)
data = pd.read_csv(
    "https://raw.githubusercontent.com/mlflow/mlflow/master/tests/datasets/winequality-white.csv",
    sep=";"  # US Portuguese CSV uses semicolon as separator
)

print(f"✅ Dataset loaded!")
print(f"   Shape: {data.shape[0]} samples × {data.shape[1]} features")
print(f"\n📊 First few rows:")
print(data.head())

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.00100,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.99400,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.99510,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
...,...,...,...,...,...,...,...,...,...,...,...,...
4893,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,6
4894,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,5
4895,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,6
4896,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,7


In [ ]:
# ============================================================================
# STEP 3: Split Data into Training and Test Sets
# ============================================================================
# We'll use:
# - 75% for training + validation (model gets to learn)
# - 25% for testing (evaluate on completely unseen data)

print("🔀 Splitting data...\n")

train, test = train_test_split(data, test_size=0.25, random_state=42)
print(f"✅ Data split:")
print(f"   Training + Validation: {len(train)} samples (75%)")
print(f"   Testing: {len(test)} samples (25%)")
print(f"\n📊 Training set sample:")
print(train.head())

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
2835,6.3,0.25,0.22,3.30,0.048,41.0,161.0,0.99256,3.16,0.50,10.5,6
1157,7.8,0.30,0.29,16.85,0.054,23.0,135.0,0.99980,3.16,0.38,9.0,6
744,7.4,0.38,0.27,7.50,0.041,24.0,160.0,0.99535,3.17,0.43,10.0,5
1448,7.4,0.16,0.49,1.20,0.055,18.0,150.0,0.99170,3.23,0.47,11.2,6
3338,7.2,0.27,0.28,15.20,0.046,6.0,41.0,0.99665,3.17,0.39,10.9,6
...,...,...,...,...,...,...,...,...,...,...,...,...
4426,6.2,0.21,0.52,6.50,0.047,28.0,123.0,0.99418,3.22,0.49,9.9,6
466,7.0,0.14,0.32,9.00,0.039,54.0,141.0,0.99560,3.22,0.43,9.4,6
3092,7.6,0.27,0.52,3.20,0.043,28.0,152.0,0.99129,3.02,0.53,11.4,6
3772,6.3,0.24,0.29,13.70,0.035,53.0,134.0,0.99567,3.17,0.38,10.6,6


In [15]:
train[["quality"]].values.ravel()

array([6, 6, 5, ..., 6, 6, 8], shape=(3673,))

In [ ]:
# ============================================================================
# STEP 4: Further Split Data and Separate Features from Target
# ============================================================================
# We need 3 datasets:
# 1. Train: Teach the model (60% of original)
# 2. Validation: Tune hyperparameters and prevent overfitting (15% of original)
# 3. Test: Final evaluation on completely unseen data (25% of original)

print("🔀 Creating train/validation/test sets...\n")

# ============================================================================
# Prepare Training Data
# ============================================================================
# x = features (all columns except quality)
# y = target (quality score, what we want to predict)
train_x = train.drop(["quality"], axis=1).values  # Convert to NumPy array
train_y = train[["quality"]].values.ravel()       # Target values (flattened to 1D)

print(f"✅ Training set: {train_x.shape[0]} samples × {train_x.shape[1]} features")

# ============================================================================
# Prepare Test Data
# ============================================================================
test_x = test.drop(["quality"], axis=1).values
test_y = test[["quality"]].values.ravel()

print(f"✅ Test set: {test_x.shape[0]} samples × {test_x.shape[1]} features")

# ============================================================================
# Split training data into train and validation
# ============================================================================
# 80% for actual training, 20% for validation during training
train_x, valid_x, train_y, valid_y = train_test_split(
    train_x, train_y, test_size=0.20, random_state=42
)

print(f"✅ Final split:")
print(f"   Train: {len(train_x)} samples")
print(f"   Validation: {len(valid_x)} samples")
print(f"   Test: {len(test_x)} samples")

# Create model signature for MLflow
signature = infer_signature(train_x, train_y)
print(f"\n✅ Model signature created")

In [17]:
np.mean(train_x,axis=0)

array([6.86621852e+00, 2.80377808e-01, 3.32597005e-01, 6.42164738e+00,
       4.55513955e-02, 3.53556841e+01, 1.38792376e+02, 9.94074221e-01,
       3.18919333e+00, 4.88396869e-01, 1.05005673e+01])

In [18]:
print(np.mean(train_x,axis=1))

[15.79594545 14.50073182 16.48442727 ... 12.97630182 17.94385455
 22.38198182]


In [ ]:
# ============================================================================
# STEP 5: Define Neural Network Training Function
# ============================================================================
# This function:
# 1. Creates a neural network with specific hyperparameters
# 2. Trains it on the training data
# 3. Evaluates it on validation data
# 4. Logs everything to MLflow
# 5. Returns the result for hyperopt to optimize

def train_model(params, epochs, train_x, train_y, valid_x, valid_y, test_x, test_y):
    """
    Build, train, and evaluate a neural network with given hyperparameters.
    
    Parameters:
    -----------
    params: dict with 'lr' (learning rate) and 'momentum' (optimizer momentum)
    epochs: number of training iterations
    train_x/y: training data
    valid_x/y: validation data (for early stopping, monitoring)
    test_x/y: test data (final evaluation)
    
    Returns:
    --------
    Dictionary with 'loss' (RMSE), 'status', and 'model'
    """
    
    # ====================================================================
    # 1. Calculate Normalization Parameters
    # ====================================================================
    # Neural networks perform better with normalized data (mean=0, std=1)
    mean = np.mean(train_x, axis=0)  # Mean of each feature
    var = np.var(train_x, axis=0)    # Variance of each feature
    
    # ====================================================================
    # 2. Define Neural Network Architecture
    # ====================================================================
    # Sequential: Stack layers one after another
    # This creates: Input → Normalize → Dense(64) → Dense(1) → Output
    
    model = keras.Sequential([
        keras.Input(shape=[train_x.shape[1]]),  # Input dim = number of wine features
        
        # Normalization layer: Scales input data to have mean=0, variance=1
        keras.layers.Normalization(mean=mean, variance=var),
        
        # Hidden layer: 64 neurons with ReLU activation
        # - 64 neurons allow the model to learn complex patterns
        # - ReLU (Rectified Linear Unit) = max(0, x) - very common activation
        keras.layers.Dense(64, activation="relu"),
        
        # Output layer: 1 neuron (predict 1 value: wine quality)
        # - No activation = linear regression head
        keras.layers.Dense(1)
    ])
    
    # ====================================================================
    # 3. Compile the Model
    # ====================================================================
    # Tell Keras how to train the model
    model.compile(
        # SGD optimizer with hyperparameters from hyperopt
        optimizer=keras.optimizers.SGD(
            learning_rate=params["lr"],      # How much to update weights each step
            momentum=params["momentum"]       # Momentum helps optimization
        ),
        loss="mean_squared_error",           # Loss function (for regression)
        metrics=[keras.metrics.RootMeanSquaredError()]  # Metric to monitor
    )
    
    # ====================================================================
    # 4. Train with MLflow Tracking
    # ====================================================================
    # nested=True: This is a sub-run under a parent run
    with mlflow.start_run(nested=True):
        print(f"   Training with lr={params['lr']:.2e}, momentum={params['momentum']:.2f}")
        
        # Train the model
        # - validation_data: Used to monitor overfitting (not for training)
        # - epochs: How many times to go through the data
        # - batch_size: How many samples per update step
        model.fit(
            train_x, train_y,
            validation_data=(valid_x, valid_y),
            epochs=epochs,
            batch_size=64,
            verbose=0  # No verbose output (training happens in background)
        )
        
        # ====================================================================
        # 5. Evaluate on Validation Data
        # ====================================================================
        # This is what hyperopt will try to minimize
        eval_result = model.evaluate(valid_x, valid_y, batch_size=64, verbose=0)
        eval_rmse = eval_result[1]  # Extract RMSE metric (lower is better)
        
        # ====================================================================
        # 6. Log to MLflow
        # ====================================================================
        # Save hyperparameters and metrics so we can track this run
        mlflow.log_params(params)
        mlflow.log_metric("validation_rmse", eval_rmse)
        
        # Save the model
        mlflow.tensorflow.log_model(model, "model", signature=signature)
        
        # ====================================================================
        # 7. Return Result for Hyperopt
        # ====================================================================
        # Hyperopt minimizes 'loss', so lower RMSE = better
        return {
            "loss": eval_rmse,        # What hyperopt will optimize (minimize)
            "status": STATUS_OK,      # Successful run
            "model": model            # Save model for later retrieval
        }

In [ ]:
# ============================================================================
# STEP 6: Define Objective Function for Hyperopt
# ============================================================================
# Hyperopt needs an objective function that it can call repeatedly
# with different hyperparameters, and returns a loss value

def objective(params):
    """
    Wrapper function for hyperopt.
    Takes hyperparameters, trains a model, and returns the loss.
    
    Hyperopt will call this function many times with different parameters,
    always trying to find parameters that minimize the loss.
    """
    
    # Train model with these hyperparameters
    result = train_model(
        params,
        epochs=10,           # Number of training epochs (might be low for quick demo)
        train_x=train_x,
        train_y=train_y,
        valid_x=valid_x,
        valid_y=valid_y,
        test_x=test_x,
        test_y=test_y,
    )
    
    return result  # Return loss, status, and model

In [ ]:
# ============================================================================
# STEP 7: Define Hyperparameter Search Space
# ============================================================================
# Hyperopt needs to know:
# 1. Which hyperparameters to optimize
# 2. What range/distribution each can take

print("🎯 Defining hyperparameter search space...\n")

space = {
    # Learning Rate (lr): How quickly the model learns
    # - hp.loguniform: Search on logarithmic scale (good for learning rates)
    # - Range: 0.00001 to 0.1 (5 orders of magnitude)
    # - Logarithmic because 0.001 ≠ 0.1 is very different (not linear)
    "lr": hp.loguniform("lr", np.log(0.00001), np.log(0.1)),
    
    # Momentum: How much to keep previous gradient direction
    # - hp.uniform: Search on linear scale
    # - Range: 0.0 to 1.0 (0=no momentum, 1=pure momentum)
    "momentum": hp.uniform("momentum", 0.0, 1.0)
}

print("✅ Hyperparameter space defined:")
print("   - Learning Rate: 1e-5 to 1e-1 (log scale)")
print("   - Momentum: 0.0 to 1.0 (linear scale)")

In [ ]:
# ============================================================================
# STEP 8: Run Hyperopt Hyperparameter Search
# ============================================================================
# Hyperopt will:
# 1. Sample different hyperparameter combinations (up to max_evals)
# 2. Train a model for each combination (nested in MLflow)
# 3. Track the validation loss
# 4. Use Tree-structured Parzen Estimator (TPE) to suggest better params
# 5. Return the BEST hyperparameters found

print("🚀 Starting Hyperopt hyperparameter search...\n")
print("⚠️  This might take 2-3 minutes (training 4 models)\n")

# Connect to MLflow and create an experiment
mlflow.set_experiment("/wine-quality")  # All runs go under this experiment

# Start parent run (encompasses all hyperopt trials)
with mlflow.start_run():
    # ====================================================================
    # Run Hyperopt Search
    # ====================================================================
    # Create Trials object to store all trial information
    trials = Trials()  # Stores every hyperopt trial's results
    
    # Run the hyperparameter optimization
    best = fmin(
        fn=objective,                    # Function to minimize (trains model)
        space=space,                     # Hyperparameter search space
        algo=tpe.suggest,                # Algorithm: Tree Parzen Estimator (smart search)
        max_evals=4,                     # Try 4 different hyperparameter combinations
        trials=trials                    # Store results in trials object
    )
    
    print(f"\n✅ Hyperopt search complete!")
    print(f"   Tested 4 different hyperparameter combinations")
    
    # ====================================================================
    # Extract Best Results
    # ====================================================================
    # trials.results is a list of all trial results
    # Each result has 'loss' and 'model'
    # Sort by loss to get the best one
    best_run = sorted(trials.results, key=lambda x: x["loss"])[0]
    
    # ====================================================================
    # Log Best Results to MLflow
    # ====================================================================
    print(f"\n📊 Best hyperparameters found:")
    print(f"   Learning Rate: {best['lr']:.2e}")
    print(f"   Momentum: {best['momentum']:.4f}")
    print(f"   Validation RMSE: {best_run['loss']:.4f}")
    
    # Log best hyperparameters
    mlflow.log_params(best)
    
    # Log best metric
    mlflow.log_metric("best_validation_rmse", best_run["loss"])
    
    # Log best model (so we can use it for inference)
    mlflow.tensorflow.log_model(
        best_run["model"],
        "best_model",
        signature=signature
    )
    
    print(f"\n🎉 Best model saved and logged to MLflow!")
    print(f"📊 View all runs at: http://127.0.0.1:5000")

Epoch 1/10                                           

 1/46 ━━━━━━━━━━━━━━━━━━━━ 49s 1s/step - loss: 36.0327 - root_mean_squared_error: 6.0027
13/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 35.1633 - root_mean_squared_error: 5.9296
24/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 34.5000 - root_mean_squared_error: 5.8732
35/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 33.9190 - root_mean_squared_error: 5.8232
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 33.3819 - root_mean_squared_error: 5.7765
46/46 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - loss: 31.1920 - root_mean_squared_error: 5.5850 - val_loss: 27.2811 - val_root_mean_squared_error: 5.2231

Epoch 2/10                                           

 1/46 ━━━━━━━━━━━━━━━━━━━━ 4s 92ms/step - loss: 25.3485 - root_mean_squared_error: 5.0347
13/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 25.9075 - root_mean_squared_error: 5.0898 
25/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 25.7350 - root_mean_squared_error: 5.0728
37/46 ━━━━━━━━━━━━━━━━━━━━ 

2026/03/19 03:56:09 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\ryaka\AppData\Local\Temp\tmpr37m2lqi\model, flavor: tensorflow). Fall back to return ['tensorflow==2.20.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 



Epoch 1/10                                                                     

 1/46 ━━━━━━━━━━━━━━━━━━━━ 51s 1s/step - loss: 37.1485 - root_mean_squared_error: 6.0950
10/46 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 37.8847 - root_mean_squared_error: 6.1550
19/46 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 37.8739 - root_mean_squared_error: 6.1541
27/46 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 37.8796 - root_mean_squared_error: 6.1546
35/46 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 37.9296 - root_mean_squared_error: 6.1587
45/46 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 37.9568 - root_mean_squared_error: 6.1609
46/46 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - loss: 37.9049 - root_mean_squared_error: 6.1567 - val_loss: 37.5135 - val_root_mean_squared_error: 6.1248

Epoch 2/10                                                                     

 1/46 ━━━━━━━━━━━━━━━━━━━━ 3s 85ms/step - loss: 37.9967 - root_mean_squared_error: 6.1641
11/46 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 38.2623 - root_mea

2026/03/19 03:56:26 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\ryaka\AppData\Local\Temp\tmp0olswveg\model, flavor: tensorflow). Fall back to return ['tensorflow==2.20.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 



Epoch 1/10                                                                     

 1/46 ━━━━━━━━━━━━━━━━━━━━ 55s 1s/step - loss: 35.7322 - root_mean_squared_error: 5.9776
10/46 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 36.8122 - root_mean_squared_error: 6.0672
20/46 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 36.3890 - root_mean_squared_error: 6.0321
32/46 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 35.5865 - root_mean_squared_error: 5.9646
41/46 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 34.9843 - root_mean_squared_error: 5.9133
46/46 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - loss: 31.8645 - root_mean_squared_error: 5.6449 - val_loss: 26.2873 - val_root_mean_squared_error: 5.1271

Epoch 2/10                                                                     

 1/46 ━━━━━━━━━━━━━━━━━━━━ 3s 89ms/step - loss: 27.7505 - root_mean_squared_error: 5.2679
13/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 25.7372 - root_mean_squared_error: 5.0727 
25/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 25.0468 - root_me

...

Epoch 1/10                                                                     

 1/46 ━━━━━━━━━━━━━━━━━━━━ 52s 1s/step - loss: 40.6156 - root_mean_squared_error: 6.3730
11/46 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 39.8249 - root_mean_squared_error: 6.3106
20/46 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 39.2649 - root_mean_squared_error: 6.2659
26/46 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 38.8744 - root_mean_squared_error: 6.2345
36/46 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 38.2112 - root_mean_squared_error: 6.1805
42/46 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 37.8489 - root_mean_squared_error: 6.1509
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 37.6139 - root_mean_squared_error: 6.1315
46/46 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - loss: 34.9901 - root_mean_squared_error: 5.9152 - val_loss: 29.5272 - val_root_mean_squared_error: 5.4339

Epoch 2/10                                                                     

 1/46 ━━━━━━━━━━━━━━━━━━━━ 4s 91ms/step - loss: 29.0440 - root_mea

2026/03/19 03:57:02 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\ryaka\AppData\Local\Temp\tmpvw3ly8vw\model, flavor: tensorflow). Fall back to return ['tensorflow==2.20.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 



100%|██████████| 4/4 [01:10<00:00, 17.55s/trial, best loss: 1.5622018575668335]


2026/03/19 03:57:12 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\ryaka\AppData\Local\Temp\tmpjv45vydp\model, flavor: tensorflow). Fall back to return ['tensorflow==2.20.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 


Best Parameters: {'lr': np.float64(0.00010949955021246393), 'momentum': np.float64(0.7501479564130394)}
Best eval rmse: 1.5622018575668335


In [12]:
train[["quality"]].values

array([[6],
       [6],
       [5],
       ...,
       [6],
       [6],
       [8]], shape=(3673, 1))

In [ ]:
# ============================================================================
# STEP 9: Load and Use the Best Model (Inference)
# ============================================================================
# Now that we've found and trained the best model, let's use it for predictions!

print("🔮 Loading best model and making predictions...\n")

# Import MLflow serving utilities
from mlflow.models import validate_serving_input, convert_input_example_to_serving_input

# Model URI: unique identifier for the best model we just saved
# Format: runs:/[run_id]/best_model
# This tells MLflow which model to load
model_uri = 'runs:/faa29eb2b96243b19aae51014b5d1124/best_model'

# ====================================================================
# Generate Serving Payload
# ====================================================================
# MLflow needs to know what format the input data should be in
# for serving (like via REST API)
serving_payload = convert_input_example_to_serving_input(test_x)

print(f"✅ Serving payload generated")

# ====================================================================
# Validate Serving Input
# ====================================================================
# Make sure the model can actually serve predictions with this format
validate_serving_input(model_uri, serving_payload)

print(f"✅ Serving input validated")

d:\anaconda\envs\mlflow_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step


array([[6.1853533],
       [7.1047106],
       [6.47804  ],
       ...,
       [6.1623287],
       [6.804157 ],
       [5.9823923]], shape=(1225, 1), dtype=float32)

In [ ]:
# ============================================================================
# STEP 10: Make Predictions with the Best Model
# ============================================================================
# Load the model and make predictions on new data

print("🤖 Using the model for predictions...\n")

# Load model as a PyFuncModel (Python Function Model)
# PyFuncModel: Standard MLflow model format that can be deployed anywhere
model_uri = 'runs:/faa29eb2b96243b19aae51014b5d1124/best_model'
loaded_model = mlflow.pyfunc.load_model(model_uri)

print(f"✅ Model loaded successfully!")

# ====================================================================
# Make Predictions
# ====================================================================
# Use the loaded model to predict wine quality on test data
predictions = loaded_model.predict(pd.DataFrame(test_x))

print(f"✅ Predictions made on {len(test_x)} test samples!")
print(f"\n📊 Sample predictions (first 10):")
print(predictions[:10])

print(f"\n📊 Prediction statistics:")
print(f"   Mean predicted quality: {predictions.mean():.2f}")
print(f"   Std predicted quality: {predictions.std():.2f}")
print(f"   Min predicted quality: {predictions.min():.2f}")
print(f"   Max predicted quality: {predictions.max():.2f}")

39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step


array([[6.1853533],
       [7.1047106],
       [6.47804  ],
       ...,
       [6.1623287],
       [6.804157 ],
       [5.9823923]], shape=(1225, 1), dtype=float32)

In [ ]:
# ============================================================================
# STEP 11: Register Model in MLflow Model Registry
# ============================================================================
# The Model Registry is like a "production catalog" for ML models
# 
# Benefits:
# - Track multiple versions of a model
# - Promote models from "staging" to "production"
# - Set up automatic deployment pipelines
# - A/B test different model versions
# - Roll back to previous versions if needed

print("📝 Registering best model in MLflow Model Registry...\n")

# Register the model with a descriptive name
# This allows other teams/services to find and use this model
mlflow.register_model(model_uri, "wine-quality")

print(f"✅ Model registered as 'wine-quality'!")
print(f"\n📊 Next steps:")
print(f"   1. Go to http://127.0.0.1:5000/models")
print(f"   2. Click on 'wine-quality' model")
print(f"   3. Promote versions through stages: Staging → Production")
print(f"   4. Use in other code with: mlflow.pyfunc.load_model('models:/wine-quality/production')")

Successfully registered model 'wine-quality'.
Created version '1' of model 'wine-quality'.


<ModelVersion: aliases=[], creation_timestamp=1773874576359, current_stage='None', description=None, last_updated_timestamp=1773874576359, name='wine-quality', run_id='faa29eb2b96243b19aae51014b5d1124', run_link=None, source='file:///d:/MLFlowStarter/2-DLFLOW/mlruns/510724209764455175/faa29eb2b96243b19aae51014b5d1124/artifacts/best_model', status='READY', status_message=None, tags={}, user_id=None, version=1>